# Global Power Plant Inventory

Starter notebook for the Kaggle dataset in this folder. It loads `power_plants.csv` with pandas and walks through quality checks and a first piece of analysis.

**Files:** `power_plants.csv`, `data_dictionary.csv`.

## 1. Load the data

In [ ]:
import pandas as pd

df = pd.read_csv("power_plants.csv", low_memory=False)
df["capacity_mw"] = pd.to_numeric(df["capacity_mw"], errors="coerce")
print(df.shape)

## 2. Shape, types and missing values

In [ ]:
import pandas as pd

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("missing_%").head(15)

## 3. Numeric summary

In [ ]:
df.describe(include='number').T.round(2).head(20)

## 4. Capacity by fuel group

In [ ]:
(df.groupby("fuel_group")["capacity_mw"].sum() / 1000
 ).round(1).sort_values(ascending=False).to_frame("capacity_gw")

## 5. Where the capacity sits

Top 15 countries by total installed capacity.

In [ ]:
top = (df.groupby("country_long")["capacity_mw"].sum() / 1000).sort_values(ascending=False).head(15)
top.plot.barh(figsize=(8, 6))

## 6. Observed capacity factors

Only plants that report generation have a value — compare fuels.

In [ ]:
sub = df.dropna(subset=["mean_capacity_factor_pct"])
sub.boxplot(column="mean_capacity_factor_pct", by="fuel_group", figsize=(11, 5), rot=30)

## 7. Ageing fleet

In [ ]:
df.dropna(subset=["plant_age_years"]).query("plant_age_years >= 0").hist(
    column="plant_age_years", by="fuel_group", bins=30, figsize=(14, 8), sharex=True)

## 8. Renewables share of capacity by region

In [ ]:
(df.groupby(["region", "fuel_group"])["capacity_mw"].sum()
   .unstack(fill_value=0).pipe(lambda t: t.div(t.sum(axis=1), axis=0) * 100).round(1))

## Next steps

See `KAGGLE.md` for analysis ideas, caveats and provenance.